# 04 — Positional Encoding — Solution

---

In [ ]:
import sys, os, math
sys.path.insert(0, os.path.abspath('../..'))
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt
from src.models.attention import MultiHeadAttention
from src.models.embeddings import SinusoidalPositionalEncoding, LearnedPositionalEmbedding, RotaryPositionalEmbedding

torch.manual_seed(7)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Part A — Permutation Invariance Demo

In [ ]:
mha = MultiHeadAttention(d_model=32, n_heads=2)
mha.eval()

x     = torch.randn(1, 6, 32)
perm  = [3, 0, 5, 1, 4, 2]
x_p   = x[:, perm, :]

with torch.no_grad():
    out,  _ = mha(x,  x,  x)
    out_p, _ = mha(x_p, x_p, x_p)

# If attention is truly permutation-equivariant: out_p == out[:, perm, :]
expected = out[:, perm, :]
matches  = torch.allclose(expected, out_p, atol=1e-5)
print(f'Permuted output matches shuffled original: {matches}')
print('→ Attention treats the input as a SET — order is invisible without positional encoding.')

## Part B — Sinusoidal Encoding

In [ ]:
def sinusoidal_encoding(seq_len, d_model):
    pe = torch.zeros(seq_len, d_model)
    position  = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)  # (seq, 1)
    div_term  = torch.exp(
        torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
    )  # (d_model/2,)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

pe = sinusoidal_encoding(50, 64)
print('PE shape:', pe.shape)

plt.figure(figsize=(12, 4))
plt.imshow(pe.numpy().T, cmap='RdBu', aspect='auto', vmin=-1, vmax=1)
plt.xlabel('Position'); plt.ylabel('Dimension')
plt.title('Sinusoidal Positional Encoding'); plt.colorbar(); plt.tight_layout(); plt.show()

In [ ]:
# B2 — relative position via dot product
sim = pe @ pe.T

plt.figure(figsize=(6, 5))
plt.imshow(sim.numpy(), cmap='RdBu', aspect='auto')
plt.xlabel('Position j'); plt.ylabel('Position i')
plt.title('PE dot-product similarity — bands along diagonals confirm it depends only on |i-j|')
plt.colorbar(); plt.tight_layout(); plt.show()

## Parts C & D — Library Comparison & RoPE

In [ ]:
d_model, seq_len = 64, 100
sin_pe     = SinusoidalPositionalEncoding(d_model, seq_len)
learned_pe = LearnedPositionalEmbedding(seq_len, d_model)

sin_enc     = sin_pe.get_encoding(seq_len).detach().numpy()
learned_enc = learned_pe.get_encoding(seq_len).detach().numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, enc, title in zip(axes, [sin_enc, learned_enc], ['Sinusoidal (fixed)', 'Learned (random init)']):
    im = ax.imshow(enc.T, cmap='RdBu', aspect='auto', vmin=-1, vmax=1)
    ax.set_xlabel('Position'); ax.set_ylabel('Dimension'); ax.set_title(title)
    plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

# RoPE
rope = RotaryPositionalEmbedding(d_model=64, max_seq_len=256)
q = torch.randn(1, 1, 20, 64)
k = torch.randn(1, 1, 20, 64)
q_rot, k_rot = rope(q, k)
print(f'RoPE: norm preserved = {torch.allclose(q.norm(dim=-1), q_rot.norm(dim=-1), atol=1e-5)}')